# CNN and live packet signal lab

This notebook introduces two additions to the project: a validation-only 1D CNN comparison on the existing labeled flow dataset, and a separate packet-time-series demo with filtering and a convolutional autoencoder.

They answer different questions. The flow CNN predicts the existing binary label from 36 flow features. The live demo captures packet metadata, builds a time signal, and reports an anomaly score relative to a local baseline. The live score is not an attack label and is not the original HGB model.


## Part A — 1D CNN on the labeled flow dataset

Run the CNN comparison from the project root:

```bash
.venv/bin/python live_traffic_demo/flow_cnn.py \
  --data data/raw/NF-CSE-CIC-IDS2018-V2.parquet
```

The script uses the same binary caps (125,000 Benign and 17,000 attack rows), seed 43, 70/15/15 stratified split, and 36 baseline predictors. The test partition is reserved. The CNN is a small Conv1D network with two convolution layers, global average pooling, and two output logits. Logistic regression receives a training-fitted StandardScaler; the CNN receives the same standardized feature representation. The fit uses class-balanced loss weights.

**Careful interpretation:** the convolution axis here is the ordered feature columns. The 36 columns are heterogeneous flow statistics, not 36 time samples. This CNN is a deep-learning comparison, not the time-series Signals and Systems component. It did not beat HGB on validation, so HGB remains the selected model for the original binary task.


In [ ]:
from pathlib import Path
import json

result_path = Path("live_traffic_demo/flow_cnn_validation.json")
report = json.loads(result_path.read_text())
print("Validation attack F1 (source-frequency weighted)")
for model, score in report["validation_attack_f1"].items():
    print(f"{model:32} {score:.4f}")
print("\nBest CNN epoch:", report["cnn_best_epoch"])
print("Training rows:", report["split_rows"]["train"])
print("Validation rows:", report["split_rows"]["validation"])


## Part B — live packet signal demo

Start the separate local page in simulation mode:

```bash
.venv/bin/python live_traffic_demo/app.py --mode simulate
```

Open `http://127.0.0.1:8765`. This mode creates a labeled normal-like baseline and a second trace with two generated packet bursts. It lets you demonstrate the signal processing without packet-capture permissions. Its precision/recall apply only to these generated patterns.

For actual packet metadata, choose an interface that you are authorized to monitor and start the capture mode before presenting:

```bash
sudo .venv/bin/python live_traffic_demo/app.py \
  --mode live --interface wlan0 --baseline-seconds 300 --monitor-seconds 300
```

Replace `wlan0` with the lab machine's interface name. The first five minutes calibrate a local baseline; the next five minutes are monitored. Start this in advance. If the lab machine does not grant packet-capture permission, use simulation mode and explain the live adapter as the next deployment step.

The capture callback reads frame length and timestamp and discards each packet. It does not save payloads or PCAP files.


## Signals and Systems concepts in the live path

For each second, the demo records three measurements: packet count, total frame bytes, and mean packet inter-arrival time. This creates three discrete-time channels. It then applies these operations:

1. **Dynamic-range compression:** `z[n] = log(1 + x[n])` reduces the effect of very large byte counts.
2. **Causal EWMA low-pass filter:** `y[n] = αz[n] + (1−α)y[n−1]`, with `α = 0.22`. The filter smooths short fluctuations and only uses the current and past measurements.
3. **High-pass/change component:** `r[n] = z[n] − y[n]` retains deviations from the smoothed baseline.
4. **Scaling:** six channels—three smoothed signals and three residuals—are standardized using the calibration period.
5. **Sliding windows:** the CNN receives 32 time bins (32 seconds) at a time.

The dashboard plots packet rate, its low-pass estimate, and the reconstruction error. This makes the system/transformation visible during the demonstration, rather than merely naming a neural network.


### Simulation-only check

The included fixed-seed run demonstrates that the CNN can flag the deliberately generated bursts. These numbers describe the simulation, not captured traffic or attack-detection performance.


In [ ]:
simulation = json.loads(Path("live_traffic_demo/simulation_metrics.json").read_text())
print(simulation["note"])
print(f"precision={simulation['precision']:.3f}; recall={simulation['recall']:.3f}; false-positive rate={simulation['false_positive_rate']:.3f}")


## The temporal CNN

The live model is a **1D convolutional autoencoder**. Its input shape is `(batch, 6 channels, 32 time steps)`. Convolutions move along the time axis. The encoder compresses each 32-second window; the decoder reconstructs the six input channels. It is trained only on the first 80% of a baseline period. The later 20% calibrates the alert threshold.

For each later window, the detector computes mean squared reconstruction error. The threshold is set to twice the 99th percentile of calibration errors. Above threshold, the demo reports **“unusual compared with this baseline.”** It cannot say that an attack occurred or name an attack type because live traffic has no ground-truth labels.

The simulated page uses generated data to demonstrate this end-to-end. The live mode learns a new baseline from the selected interface. In both cases, this is an experimental anomaly score, not a validated intrusion-prevention system.


## What to say about the results and limits

- The flow-feature CNN reached validation attack F1 **0.9349**; HGB reached **0.9766** and remained the selected binary model.
- The flow CNN uses feature-axis convolution, so do not describe it as learning temporal traffic behavior.
- The live CNN is a different model: an unsupervised temporal autoencoder that scores unusual signal windows.
- Simulation results demonstrate the processing chain only; they are not measured on real attacks.
- Live anomaly scores are relative to the chosen baseline. Benign workload changes may trigger alerts; attacks that resemble baseline traffic may not.
- The live capture mode requires packet-capture permission and a clean baseline period. The original static website remains separate and unchanged.

This extension gives the project an honest S&S story: timestamped packet observations become discrete-time signals, then compression, causal filtering, residual analysis, scaling, windowing, and convolutional modeling are applied before scoring.
